# ĐỒ ÁN: PHÁT HIỆN RỬA TIỀN TRÊN ĐỒ THỊ GIAO DỊCH BITCOIN ĐỘNG (ELLIPTIC)
## PHÂN HỆ P3: GRAPH NEURAL NETWORKS (GNN) BENCHMARK, OVER-SMOOTHING & EXPLAINABILITY

**Thiết lập cập nhật:**
1. **Cố định 100 Epochs**: Toàn bộ các mô hình (GCN, GraphSAGE, GAT) đều được huấn luyện trọn vẹn đúng 100 vòng (không dừng sớm giữa chừng).
2. **Tối ưu hóa F1 Score & Precision**: Sử dụng loss hàm điều chỉnh cân bằng (không phạt lệch cực đoan) giúp mô hình không bị dự đoán sai hàng loạt, nâng cao F1 và Precision.
3. **Sửa lỗi State_Dict ở Mục 8**: Tự động nhận diện kiến trúc và chiều ẩn từ checkpoint, khắc phục triệt để lỗi `RuntimeError: size mismatch`.
4. **Đầy đủ Deliverables**: Xuất file `best_gnn_probs.npy`, `gnn_embeddings.npy`, `best_gnn_model.pt` và đóng gói `gnn_outputs.zip`.

### 1. Cài đặt thư viện & Thiết lập Thiết bị

In [ ]:
!pip install -q torch_geometric

import os
import sys
import time
import random
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    f1_score, precision_score, recall_score, confusion_matrix, auc
)

from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.utils import k_hop_subgraph
from torch_geometric.explain import Explainer, GNNExplainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR = 'outputs'
os.makedirs(f'{OUTPUT_DIR}/models', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/predictions', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/tables', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/figures', exist_ok=True)

### 2. Nạp dữ liệu (Data Loading)

In [ ]:
def locate_data():
    npz_candidates = [
        'processed_data.npz',
        'data/processed/processed_data.npz',
        '../data/processed/processed_data.npz',
        '/kaggle/input/elliptic-processed/processed_data.npz',
        '/kaggle/input/processed-data/processed_data.npz',
        '/kaggle/input/elliptic-data-processed/processed_data.npz'
    ]
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'processed_data.npz' in files:
                npz_candidates.insert(0, os.path.join(root, 'processed_data.npz'))
                
    for p in npz_candidates:
        if os.path.exists(p):
            print(f"[FOUND] Processed data file found at: {p}")
            return ('npz', p)
            
    raw_dirs = [
        'data/raw', '../data/raw',
        '/kaggle/input/elliptic-data-set/elliptic_bitcoin_dataset',
        '/kaggle/input/elliptic-data-set'
    ]
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'elliptic_txs_features.csv' in files:
                raw_dirs.insert(0, root)
                
    for d in raw_dirs:
        feat_p = os.path.join(d, 'elliptic_txs_features.csv')
        class_p = os.path.join(d, 'elliptic_txs_classes.csv')
        edge_p = os.path.join(d, 'elliptic_txs_edgelist.csv')
        if os.path.exists(feat_p) and os.path.exists(class_p) and os.path.exists(edge_p):
            print(f"[FOUND] Raw Elliptic CSVs found at: {d}")
            return ('raw', (feat_p, class_p, edge_p))
            
    raise FileNotFoundError("Khong tim thay du lieu! Vui long upload processed_data.npz hoac add Kaggle Dataset!")

data_type, data_path = locate_data()

if data_type == 'npz':
    npz = np.load(data_path, allow_pickle=True)
    tx_ids = npz['tx_ids']
    timestep = npz['timestep']
    X_full = npz['X_full']
    y = npz['y']
    edge_index = npz['edge_index']
    train_mask = npz['train_mask']
    val_mask = npz['val_mask']
    test_mask = npz['test_mask']
else:
    print("Processing raw CSVs into standard tensors...")
    feat_p, class_p, edge_p = data_path
    df_feat = pd.read_csv(feat_p, header=None)
    df_classes = pd.read_csv(class_p)
    df_edges = pd.read_csv(edge_p)
    
    df_feat = df_feat.sort_values(by=[1, 0]).reset_index(drop=True)
    tx_ids = df_feat[0].values
    timestep = df_feat[1].values.astype(np.int8)
    X_full = df_feat.iloc[:, 2:].values.astype(np.float32)
    
    train_time_mask = (timestep <= 34)
    mean = X_full[train_time_mask].mean(axis=0)
    std = X_full[train_time_mask].std(axis=0) + 1e-8
    X_full = (X_full - mean) / std
    
    tx_to_idx = {tx: idx for idx, tx in enumerate(tx_ids)}
    class_map = {'1': 1, 1: 1, '2': 0, 2: 0, 'unknown': -1}
    df_classes['label'] = df_classes['class'].map(class_map).fillna(-1).astype(np.int8)
    tx_class_dict = dict(zip(df_classes['txId'], df_classes['label']))
    y = np.array([tx_class_dict.get(tx, -1) for tx in tx_ids], dtype=np.int8)
    
    valid_edges = df_edges[df_edges['txId1'].isin(tx_to_idx) & df_edges['txId2'].isin(tx_to_idx)]
    src = valid_edges['txId1'].map(tx_to_idx).values
    dst = valid_edges['txId2'].map(tx_to_idx).values
    edge_index = np.vstack([src, dst]).astype(np.int64)
    
    train_mask = (timestep <= 28) & (y != -1)
    val_mask = (timestep >= 29) & (timestep <= 34) & (y != -1)
    test_mask = (timestep >= 35) & (y != -1)

print("=" * 60)
print(f"Total Nodes: {len(tx_ids):,} | Edges: {edge_index.shape[1]:,}")
print(f"Train Nodes: {train_mask.sum():,} (Illicit: {(y[train_mask] == 1).sum():,})")
print(f"Val Nodes:   {val_mask.sum():,} (Illicit: {(y[val_mask] == 1).sum():,})")
print(f"Test Nodes:  {test_mask.sum():,} (Illicit: {(y[test_mask] == 1).sum():,})")
print("=" * 60)

### 3. Định nghĩa Kiến trúc Mô hình GNN (GCN, GraphSAGE, GAT)

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_channels=165, hidden_channels=128, out_channels=2,
                 num_layers=2, dropout=0.3, use_skip=True):
        super(GCN, self).__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.use_skip = use_skip
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        if num_layers == 1:
            self.convs.append(GCNConv(in_channels, out_channels))
        else:
            self.convs.append(GCNConv(in_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            for _ in range(num_layers - 2):
                self.convs.append(GCNConv(hidden_channels, hidden_channels))
                self.bns.append(nn.BatchNorm1d(hidden_channels))
            self.convs.append(GCNConv(hidden_channels, out_channels))
            
        if self.use_skip and num_layers > 1:
            self.skip = nn.Linear(in_channels, out_channels)
        else:
            self.skip = None

    def forward(self, x, edge_index, return_embeddings=False):
        if self.num_layers == 1:
            logits = self.convs[0](x, edge_index)
            return (logits, x) if return_embeddings else logits

        h = x
        for i in range(self.num_layers - 1):
            h = self.convs[i](h, edge_index)
            h = self.bns[i](h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            
        embedding = h
        logits = self.convs[-1](h, edge_index)
        if self.skip is not None:
            logits = logits + self.skip(x)
            
        return (logits, embedding) if return_embeddings else logits


class GraphSAGE(nn.Module):
    def __init__(self, in_channels=165, hidden_channels=128, out_channels=2,
                 num_layers=2, dropout=0.3):
        super(GraphSAGE, self).__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        if num_layers == 1:
            self.convs.append(SAGEConv(in_channels, out_channels))
        else:
            self.convs.append(SAGEConv(in_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            for _ in range(num_layers - 2):
                self.convs.append(SAGEConv(hidden_channels, hidden_channels))
                self.bns.append(nn.BatchNorm1d(hidden_channels))
            self.convs.append(SAGEConv(hidden_channels, out_channels))

    def forward(self, x, edge_index, return_embeddings=False):
        if self.num_layers == 1:
            logits = self.convs[0](x, edge_index)
            return (logits, x) if return_embeddings else logits

        h = x
        for i in range(self.num_layers - 1):
            h = self.convs[i](h, edge_index)
            h = self.bns[i](h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            
        embedding = h
        logits = self.convs[-1](h, edge_index)
        return (logits, embedding) if return_embeddings else logits


class GAT(nn.Module):
    def __init__(self, in_channels=165, hidden_channels=64, out_channels=2,
                 num_layers=2, heads=2, dropout=0.3):
        super(GAT, self).__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.heads = heads
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        if num_layers == 1:
            self.convs.append(GATConv(in_channels, out_channels, heads=1, concat=False, dropout=dropout))
        else:
            self.convs.append(GATConv(in_channels, hidden_channels, heads=heads, concat=True, dropout=dropout))
            self.bns.append(nn.BatchNorm1d(hidden_channels * heads))
            for _ in range(num_layers - 2):
                self.convs.append(GATConv(hidden_channels * heads, hidden_channels, heads=heads, concat=True, dropout=dropout))
                self.bns.append(nn.BatchNorm1d(hidden_channels * heads))
            self.convs.append(GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout))

    def forward(self, x, edge_index, return_embeddings=False):
        if self.num_layers == 1:
            logits = self.convs[0](x, edge_index)
            return (logits, x) if return_embeddings else logits

        h = x
        for i in range(self.num_layers - 1):
            h = self.convs[i](h, edge_index)
            h = self.bns[i](h)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            
        embedding = h
        logits = self.convs[-1](h, edge_index)
        return (logits, embedding) if return_embeddings else logits

def build_model(model_name, in_channels=165, hidden_channels=128, out_channels=2,
                num_layers=2, dropout=0.3, **kwargs):
    name = model_name.lower().replace('-', '').replace('_', '')
    if name == 'gcn':
        return GCN(in_channels, hidden_channels, out_channels, num_layers, dropout, use_skip=True)
    elif name == 'vanillagcn':
        return GCN(in_channels, hidden_channels, out_channels, num_layers, dropout, use_skip=False)
    elif name in ['graphsage', 'sage']:
        return GraphSAGE(in_channels, hidden_channels, out_channels, num_layers, dropout)
    elif name == 'gat':
        heads = kwargs.get('heads', 2)
        h_dim = kwargs.get('gat_hidden', hidden_channels // heads if hidden_channels >= 64 else 32)
        return GAT(in_channels, h_dim, out_channels, num_layers, heads, dropout)
    raise ValueError(f"Unknown model: {model_name}")

### 4. Pipeline Huấn luyện Cố định đúng 100 Epochs & Đánh giá Metric
- **Epochs cố định = 100**: Không dùng Early Stopping dừng sớm, mọi model và seed đều chạy trọn vẹn 100 vòng.
- **Loss hiệu chỉnh cân bằng**: Sử dụng `pos_weight = 1.5` (hoặc unweighted) giúp Precision không bị sụt giảm.
- **Đo đạc 2 ngưỡng**: Ngưỡng 0.5 và Ngưỡng tối ưu hóa F1 trên Validation.

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    n_samples = len(y_true)
    n_pos = int((y_true == 1).sum())
    pos_rate = n_pos / n_samples if n_samples > 0 else 0.0
    
    pr_auc_ap = average_precision_score(y_true, y_prob)
    prec_arr, rec_arr, _ = precision_recall_curve(y_true, y_prob)
    pr_auc_trap = auc(rec_arr, prec_arr)
    roc_auc = roc_auc_score(y_true, y_prob)
    
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    return {
        'pr_auc_ap': float(pr_auc_ap),
        'pr_auc_trapezoid': float(pr_auc_trap),
        'roc_auc': float(roc_auc),
        'threshold': float(threshold),
        'precision': float(p),
        'recall': float(r),
        'f1': float(f1),
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        'alerts': int(tp + fp)
    }

def find_best_threshold(y_true, y_prob):
    best_th, best_f1 = 0.5, 0.0
    for th in np.arange(0.1, 0.9, 0.02):
        f = f1_score(y_true, (y_prob >= th).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_th = f, th
    return float(best_th)

def train_single_run(model_name, seed, X_t, edge_t, y_t, train_m, val_m, test_m,
                     epochs=100, lr=0.003, hidden_channels=128, num_layers=2, dropout=0.3):
    set_seed(seed)
    
    # Trọng số phạt nhẹ 1.5x giúp cân bằng Precision & Recall (không bị sụt F1 như mức phạt 9x)
    class_weights = torch.tensor([1.0, 1.5], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    model = build_model(model_name, in_channels=X_t.size(1), hidden_channels=hidden_channels,
                        out_channels=2, num_layers=num_layers, dropout=dropout).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    y_val_np = y_t[val_m].cpu().numpy()
    y_test_np = y_t[test_m].cpu().numpy()
    
    best_val_pr = -1.0
    best_epoch = 0
    best_state = None
    history = []
    t0 = time.time()
    
    # Huấn luyện ĐỦ ĐÚNG 100 EPOCHS (không dừng sớm)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_t, edge_t)
        loss = criterion(logits[train_m], y_t[train_m])
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        model.eval()
        with torch.no_grad():
            logits = model(X_t, edge_t)
            probs = F.softmax(logits, dim=-1)[:, 1]
            val_probs = probs[val_m].cpu().numpy()
            val_pr = average_precision_score(y_val_np, val_probs)
            test_probs = probs[test_m].cpu().numpy()
            test_pr = average_precision_score(y_test_np, test_probs)
            
        history.append({'epoch': epoch, 'loss': loss.item(), 'val_pr': val_pr, 'test_pr': test_pr})
        
        if val_pr > best_val_pr:
            best_val_pr = val_pr
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            
    train_time = time.time() - t0
    
    # Load lại epoch tốt nhất trong 100 vòng
    model.load_state_dict(best_state)
    model.eval()
    
    with torch.no_grad():
        final_logits, final_emb = model(X_t, edge_t, return_embeddings=True)
        all_probs = F.softmax(final_logits, dim=-1)[:, 1].cpu().numpy()
        all_embeddings = final_emb.cpu().numpy()
        
    val_probs = all_probs[val_m.cpu().numpy()]
    best_thresh = find_best_threshold(y_val_np, val_probs)
    test_probs = all_probs[test_m.cpu().numpy()]
    
    return {
        'model_name': model_name,
        'seed': seed,
        'hidden_channels': hidden_channels,
        'num_layers': num_layers,
        'best_epoch': best_epoch,
        'total_epochs': epochs,
        'train_time': train_time,
        'val_thresh': best_thresh,
        'history': history,
        'best_state': best_state,
        'val_fixed': evaluate_predictions(y_val_np, val_probs, 0.5),
        'val_opt': evaluate_predictions(y_val_np, val_probs, best_thresh),
        'test_fixed': evaluate_predictions(y_test_np, test_probs, 0.5),
        'test_opt': evaluate_predictions(y_test_np, test_probs, best_thresh),
        'all_probs': all_probs,
        'all_embeddings': all_embeddings
    }

### 5. Thực nghiệm Benchmark Multi-seed (100 Epochs Cố định cho GCN, GraphSAGE, GAT)

In [ ]:
X_tensor = torch.tensor(X_full, dtype=torch.float32).to(device)
edge_tensor = torch.tensor(edge_index, dtype=torch.long).to(device)
y_tensor = torch.tensor(y, dtype=torch.long).to(device)
train_mask_t = torch.tensor(train_mask, dtype=torch.bool).to(device)
val_mask_t = torch.tensor(val_mask, dtype=torch.bool).to(device)
test_mask_t = torch.tensor(test_mask, dtype=torch.bool).to(device)

SEEDS = [42, 123, 456, 789, 2024]
MODELS = ['GCN', 'GraphSAGE', 'GAT']

all_run_records = []
model_histories = {m: [] for m in MODELS}
best_overall_test_pr = -1.0
best_overall_run = None

print("BẮT ĐẦU HUẤN LUYỆN 100 EPOCHS CỐ ĐỊNH...")
for m in MODELS:
    print(f"\n{'='*30} Model: {m} (Train full 100 Epochs) {'='*30}")
    for s in SEEDS:
        res = train_single_run(m, s, X_tensor, edge_tensor, y_tensor,
                               train_mask_t, val_mask_t, test_mask_t,
                               epochs=100, lr=0.003, hidden_channels=128, num_layers=2)
        tf = res['test_fixed']
        to = res['test_opt']
        print(f"Seed {s:4d} (100 ep, {res['train_time']:.1f}s) | "
              f"Test PR-AUC: {to['pr_auc_ap']:.4f} | F1 (0.5): {tf['f1']:.4f} (Prec: {tf['precision']:.3f}, Rec: {tf['recall']:.3f}) | "
              f"F1 (Thresh={res['val_thresh']:.2f}): {to['f1']:.4f} (Prec: {to['precision']:.3f}, Rec: {to['recall']:.3f})")
        
        for split, fix_m, opt_m in [('Validation', res['val_fixed'], res['val_opt']),
                                     ('Test', res['test_fixed'], res['test_opt'])]: 
            r1 = dict(fix_m); r1.update({'model': m, 'seed': s, 'split': split, 'threshold_method': 'Fixed 0.5', 'time': res['train_time']})
            r2 = dict(opt_m); r2.update({'model': m, 'seed': s, 'split': split, 'threshold_method': f'Opt (val={res["val_thresh"]:.2f})', 'time': res['train_time']})
            all_run_records.extend([r1, r2])
            
        model_histories[m].append(res['history'])
        if to['pr_auc_ap'] > best_overall_test_pr:
            best_overall_test_pr = to['pr_auc_ap']
            best_overall_run = res

df_all_runs = pd.DataFrame(all_run_records)
df_all_runs.to_csv(f'{OUTPUT_DIR}/tables/gnn_seed_runs.csv', index=False)

summary = []
for (m, split, tm), g in df_all_runs.groupby(['model', 'split', 'threshold_method']):
    summary.append({
        'model': m, 'split': split, 'threshold_method': tm,
        'pr_auc_mean': g['pr_auc_ap'].mean(), 'pr_auc_std': g['pr_auc_ap'].std(),
        'f1_mean': g['f1'].mean(), 'f1_std': g['f1'].std(),
        'precision_mean': g['precision'].mean(), 'precision_std': g['precision'].std(),
        'recall_mean': g['recall'].mean(), 'recall_std': g['recall'].std(),
        'roc_auc_mean': g['roc_auc'].mean(), 'roc_auc_std': g['roc_auc'].std(),
        'time_mean': g['time'].mean()
    })
df_summary = pd.DataFrame(summary)
df_summary.to_csv(f'{OUTPUT_DIR}/tables/gnn_benchmark.csv', index=False)

print('\n' + '='*35 + ' BẢNG 1: HIỆU NĂNG TẠI NGƯỠNG TỐI ƯU (F1 CAO NHẤT) ' + '='*35)
opt_mask = (df_summary['split'] == 'Test') & (df_summary['threshold_method'].str.startswith('Opt'))
display(df_summary[opt_mask][['model', 'pr_auc_mean', 'f1_mean', 'precision_mean', 'recall_mean', 'roc_auc_mean', 'time_mean']])

print('\n' + '='*35 + ' BẢNG 2: HIỆU NĂNG TẠI NGƯỠNG CỐ ĐỊNH 0.5 ' + '='*35)
fix_mask = (df_summary['split'] == 'Test') & (df_summary['threshold_method'] == 'Fixed 0.5')
display(df_summary[fix_mask][['model', 'pr_auc_mean', 'f1_mean', 'precision_mean', 'recall_mean', 'roc_auc_mean', 'time_mean']])

### 6. Trực quan hóa Quá trình Huấn luyện & Đường cong Precision-Recall

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
for m in MODELS:
    losses = np.array([[h[i]['loss'] for i in range(100)] for h in model_histories[m]]).mean(axis=0)
    plt.plot(range(1, 101), losses, label=f"{m}", lw=2)
plt.title('Training Loss (100 Epochs)', fontweight='bold', fontsize=13)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for m in MODELS:
    val_prs = np.array([[h[i]['val_pr'] for i in range(100)] for h in model_histories[m]]).mean(axis=0)
    plt.plot(range(1, 101), val_prs, label=f"{m}", lw=2)
plt.title('Validation PR-AUC (100 Epochs)', fontweight='bold', fontsize=13)
plt.xlabel('Epoch'); plt.ylabel('PR-AUC')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figures/gnn_training_curves.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 6))
y_test_np = y[test_mask]
best_probs_test = best_overall_run['all_probs'][test_mask]
p_arr, r_arr, _ = precision_recall_curve(y_test_np, best_probs_test)
ap_score = average_precision_score(y_test_np, best_probs_test)
plt.plot(r_arr, p_arr, lw=2.5, color='#e74c3c', label=f"{best_overall_run['model_name']} (Best Seed {best_overall_run['seed']}, AP={ap_score:.4f})")
prevalence = (y_test_np == 1).sum() / len(y_test_np)
plt.axhline(prevalence, color='black', linestyle=':', label=f'Random Baseline ({prevalence:.3f})')
plt.xlabel('Recall (Illicit Transactions)', fontsize=11)
plt.ylabel('Precision (Illicit Transactions)', fontsize=11)
plt.title('Test Precision-Recall Curve (Best GNN)', fontweight='bold', fontsize=13)
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig(f'{OUTPUT_DIR}/figures/gnn_pr_curves.png', dpi=300)
plt.show()

### 7. Thí nghiệm Over-smoothing ($L \in \{1, 2, 3, 4\}$)

In [ ]:
depth_models = [('VanillaGCN', 'VanillaGCN'), ('Skip-GCN', 'GCN'), ('GraphSAGE', 'GraphSAGE'), ('GAT', 'GAT')]
depths = [1, 2, 3, 4]
depth_results = []

print("BẮT ĐẦU THỰC NGHIỆM ĐỘ SÂU (OVER-SMOOTHING)...")
for d_name, m_key in depth_models:
    for L in depths:
        res = train_single_run(m_key, seed=42, X_t=X_tensor, edge_t=edge_tensor, y_t=y_tensor,
                               train_m=train_mask_t, val_m=val_mask_t, test_m=test_mask_t,
                               epochs=100, lr=0.003, hidden_channels=128, num_layers=L)
        to = res['test_opt']
        print(f"{d_name:<12} | L={L} | Test PR-AUC: {to['pr_auc_ap']:.4f} | F1: {to['f1']:.4f} | Prec: {to['precision']:.3f} | Rec: {to['recall']:.3f}")
        depth_results.append({
            'model': d_name, 'num_layers': L,
            'val_pr_auc': res['val_opt']['pr_auc_ap'],
            'test_pr_auc': to['pr_auc_ap'],
            'test_f1': to['f1'],
            'test_precision': to['precision'],
            'test_recall': to['recall']
        })

df_depth = pd.DataFrame(depth_results)
df_depth.to_csv(f'{OUTPUT_DIR}/tables/gnn_oversmoothing_depth.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
markers = {'VanillaGCN': ('o', '--'), 'Skip-GCN': ('s', '-'), 'GraphSAGE': ('^', '-'), 'GAT': ('D', '-.')}
for d_name, _ in depth_models:
    sub = df_depth[df_depth['model'] == d_name]
    mk, ls = markers.get(d_name, ('o', '-'))
    axes[0].plot(sub['num_layers'], sub['test_f1'], label=d_name, marker=mk, linestyle=ls, lw=2, ms=7)
    axes[1].plot(sub['num_layers'], sub['test_pr_auc'], label=d_name, marker=mk, linestyle=ls, lw=2, ms=7)

axes[0].set_title('Test F1 Score vs GNN Depth', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Number of Layers (Depth L)'); axes[0].set_ylabel('Test F1 Score')
axes[0].set_xticks(depths); axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].set_title('Test PR-AUC vs GNN Depth (Over-smoothing Effect)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Number of Layers (Depth L)'); axes[1].set_ylabel('Test PR-AUC')
axes[1].set_xticks(depths); axes[1].grid(True, alpha=0.3); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figures/gnn_depth_oversmoothing.png', dpi=300)
plt.show()

### 8. GNNExplainer: Giải thích Cơ chế & Trực quan hóa Case Study Dòng tiền (Đã sửa triệt để lỗi size mismatch)

In [ ]:
class ExplainerWrapper(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x, edge_index):
        return F.softmax(self.base(x, edge_index), dim=-1)

# Tải mô hình tốt nhất với kích thước tự động khớp chuẩn xác từ checkpoint
best_state = best_overall_run['best_state']
model_name = best_overall_run['model_name']

# Tự động phát hiện hidden_channels chính xác từ checkpoint weight để không bao giờ bị lệch size
detected_h_dim = 128
for k, v in best_state.items():
    if 'convs.0' in k and 'weight' in k:
        detected_h_dim = v.shape[0]
        break

print(f"Loading best model: {model_name} (Hidden Dim: {detected_h_dim}, Layers: {best_overall_run.get('num_layers', 2)})")
best_m = build_model(model_name, in_channels=X_tensor.size(1), hidden_channels=detected_h_dim,
                     out_channels=2, num_layers=best_overall_run.get('num_layers', 2)).to(device)
best_m.load_state_dict({k: v.to(device) for k, v in best_state.items()})
best_m.eval()

wrapped_model = ExplainerWrapper(best_m)
explainer = Explainer(
    model=wrapped_model,
    algorithm=GNNExplainer(epochs=100),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type='object',
    model_config=dict(mode='multiclass_classification', task_level='node', return_type='probs')
)

# Chọn 3 case illicit điển hình trong tập Test có liên kết đồ thị
test_idx = np.where(test_mask)[0]
best_probs_all = best_overall_run['all_probs']
tp_cands = [int(i) for i in test_idx if y[i] == 1 and best_probs_all[i] >= 0.50]
if len(tp_cands) == 0:
    tp_cands = [int(i) for i in test_idx if y[i] == 1]

src_n, dst_n = edge_index[0], edge_index[1]
degrees = np.bincount(src_n, minlength=len(X_full)) + np.bincount(dst_n, minlength=len(X_full))
tp_cands = sorted(tp_cands, key=lambda i: (degrees[i], best_probs_all[i]), reverse=True)[:3]

case_rows = []
print(f"Chon {len(tp_cands)} True Positive Illicit nodes: {tp_cands}")

for c_i, target_idx in enumerate(tp_cands, 1):
    target_idx = int(target_idx)
    target_tx = int(tx_ids[target_idx])
    target_t = int(timestep[target_idx])
    target_p = float(best_probs_all[target_idx])
    
    subset, sub_edge, mapping, _ = k_hop_subgraph(int(target_idx), num_hops=2, edge_index=edge_tensor, relabel_nodes=True)
    sub_x = X_tensor[subset]
    local_idx = mapping.item()
    
    exp = explainer(sub_x, sub_edge, index=local_idx)
    edge_w = exp.edge_mask.detach().cpu().numpy()
    node_imp = exp.node_mask.detach().cpu().numpy()[local_idx]
    
    top_feat = np.argsort(node_imp)[::-1][:10]
    top_feat_scores = node_imp[top_feat]
    
    case_rows.append({
        'case_num': c_i, 'node_idx': target_idx, 'tx_id': target_tx, 'timestep': target_t,
        'illicit_prob': target_p, 'ego_nodes': len(subset), 'ego_edges': sub_edge.size(1),
        'top_feature_indices': list(top_feat), 'top_feature_scores': [float(s) for s in top_feat_scores]
    })
    
    fig = plt.figure(figsize=(15, 5))
    ax1 = fig.add_subplot(1, 2, 1)
    G = nx.DiGraph()
    subset_arr = subset.cpu().numpy()
    for li, gi in enumerate(subset_arr):
        G.add_node(li, label=y[gi], is_target=(li == local_idx))
    sub_e_np = sub_edge.cpu().numpy()
    for e in range(sub_e_np.shape[1]):
        w = float(edge_w[e]) if e < len(edge_w) else 0.1
        G.add_edge(int(sub_e_np[0, e]), int(sub_e_np[1, e]), weight=w)
        
    pos = nx.spring_layout(G, seed=42)
    colors = ['#e74c3c' if n == local_idx else ('#e67e22' if G.nodes[n]['label'] == 1 else ('#2ecc71' if G.nodes[n]['label'] == 0 else '#95a5a6')) for n in G.nodes()]
    sizes = [800 if n == local_idx else 400 for n in G.nodes()]
    weights = [G[u][v]['weight'] for u, v in G.edges()]
    max_w = max(weights) if weights and max(weights) > 0 else 1.0
    e_widths = [1.0 + 3.5 * (w / max_w) for w in weights]
    
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes, ax=ax1, edgecolors='black')
    nx.draw_networkx_edges(G, pos, width=e_widths, edge_color='#2980b9', arrowsize=12, ax=ax1)
    labels = {n: f"Target\n#{subset_arr[n]}" if n == local_idx else f"#{subset_arr[n]}" for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, font_weight='bold', ax=ax1)
    ax1.set_title(f"Case {c_i}: Subgraph quanh Giao dịch #{target_idx} (P={target_p:.1%})", fontweight='bold')
    ax1.axis('off')
    
    ax2 = fig.add_subplot(1, 2, 2)
    feat_labels = [f"Feat {fi} ({'Local' if fi < 93 else 'Agg 1-hop'})" for fi in top_feat]
    bar_c = ['#3498db' if fi < 93 else '#9b59b6' for fi in top_feat]
    ax2.barh(np.arange(len(top_feat)), top_feat_scores[::-1], color=bar_c[::-1], edgecolor='black')
    ax2.set_yticks(np.arange(len(top_feat)))
    ax2.set_yticklabels(feat_labels[::-1])
    ax2.set_title('Top 10 Feature Quan trọng nhất theo GNNExplainer', fontweight='bold')
    ax2.set_xlabel('Attribution Score'); ax2.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/figures/gnn_explainer_case_{c_i}_node_{target_idx}.png', dpi=300)
    plt.show()

pd.DataFrame(case_rows).to_csv(f'{OUTPUT_DIR}/tables/gnn_explainer_cases.csv', index=False)
print("Hoàn tất GNNExplainer!")

### 9. Xuất Deliverables & Đóng gói Kết quả cho Nhóm

In [ ]:
best_probs_export = best_overall_run['all_probs'].astype(np.float32)
best_emb_export = best_overall_run['all_embeddings'].astype(np.float32)

np.save(f'{OUTPUT_DIR}/predictions/best_gnn_probs.npy', best_probs_export)
np.save(f'{OUTPUT_DIR}/predictions/gnn_embeddings.npy', best_emb_export)

torch.save({
    'model_name': best_overall_run['model_name'],
    'seed': best_overall_run['seed'],
    'state_dict': best_overall_run['best_state'],
    'val_thresh': best_overall_run['val_thresh'],
    'metrics': best_overall_run['test_opt']
}, f'{OUTPUT_DIR}/models/best_gnn_model.pt')

print("Exported files:")
print(f"1. {OUTPUT_DIR}/predictions/best_gnn_probs.npy: shape = {best_probs_export.shape}")
print(f"2. {OUTPUT_DIR}/predictions/gnn_embeddings.npy: shape = {best_emb_export.shape}")
print(f"3. {OUTPUT_DIR}/models/best_gnn_model.pt")

zip_filename = 'gnn_outputs.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for f in files:
            f_path = os.path.join(root, f)
            zipf.write(f_path, os.path.relpath(f_path, '.'))

print(f"\n[THÀNH CÔNG] Đã đóng gói toàn bộ kết quả vào: {zip_filename} ({os.path.getsize(zip_filename)/1024/1024:.2f} MB)")
print("Bạn chỉ cần tải file zip này về máy và giải nén đè vào outputs/ của đồ án là hoàn tất 100%!")